In [1]:
from glob import glob
import json
import os
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict
from tqdm import tqdm
from multiprocess import Pool
import itertools

def new_path_embedding(f):
    splitted = f.split('/')
    folder = f.split('/')[0]
    folder = folder + '_embedding'
    new_f = os.path.join(folder, '/'.join(splitted[1:]))
    new_f = new_f.replace('.mp3', '.npy').replace('.wav', '.npy')
    return new_f

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [2]:
files = glob('Multilingual-TTS/*/*.parquet')
len(files)

344

In [3]:
!rm -rf permutate-tts-speaker
!mkdir permutate-tts-speaker

In [4]:
import random

def loop(files):
    files, index = files
    
    for file in tqdm(files):
        f = file.split('/')[1]
        filename = f'permutate-tts-speaker/{f}.json'

        try:
            with open(filename) as fopen:
                json.load(fopen)
                continue
        except:
            pass
            
        df = pd.read_parquet(file)

        speakers = defaultdict(list)
        for i in range(len(df)):
            row = df.iloc[i]
            audio_path = row['audio_filename']
            new_path = new_path_embedding(audio_path)
            if not os.path.exists(new_path):
                continue
            speakers[row['speaker']].append({
                'audio': audio_path,
                'v': new_path,
                'transcription': row['text'],
            })

        data = []
        for k, v in speakers.items():
            audio_files = [r['v'] for r in v]
            vectors = []
            for f in audio_files:
                vectors.append(np.load(f))
        
            cosine = cosine_similarity(np.stack(vectors))
            data_ = []
            for n in range(len(v)):
                for m in range(n + 1, len(v)):  # upper triangle only, avoids (m,n) duplicates
                    l = v[n]
                    r = v[m]
                    if l['transcription'] == r['transcription']:
                        continue

                    if len(l['transcription']) < 5 or len(r['transcription']) < 5:
                        continue
            
                    if cosine[n, m] < 0.88:
                        continue
            
                    data_.append({
                        'reference_audio': l['audio'],
                        'reference_text': l['transcription'],
                        'target_audio': r['audio'],
                        'target_text': r['transcription'],
                        'source': file.split('/')[1],
                    })
                    data_.append({  # optional: swap direction
                        'reference_audio': r['audio'],
                        'reference_text': r['transcription'],
                        'target_audio': l['audio'],
                        'target_text': l['transcription'],
                        'source': file.split('/')[1],
                    })
            if len(data_) > 50000:
                data_ = random.sample(data_, 50000)
                
            data.extend(data_)

        with open(filename, 'w') as fopen:
            json.dump(data, fopen)

In [5]:
data = loop((files[:1], 0))

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:22<00:00, 22.61s/it]


In [6]:
def check(files):
    files, _ = files
    filtered = []
    for file in tqdm(files):
        f = file.split('/')[1]
        filename = f'permutate-tts-speaker/{f}.json'
        try:
            with open(filename) as fopen:
                d = json.load(fopen)
                continue
        except:
            pass
        filtered.append(file)
    return filtered

In [1]:
filtered = multiprocessing(files, check, cores = 20)
len(filtered)

In [2]:
multiprocessing(filtered, loop, cores = min(100, len(filtered)), returned=False)

In [3]:
!du -hs permutate-tts-speaker

113G	permutate-tts-speaker
